# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1
...,...,...
8525,any enjoyment will be hinge from a personal th...,0
8526,if legendary shlockmeister ed wood had ever ma...,0
8527,hardly a nuanced portrait of a young woman's b...,0
8528,"interminably bleak , to say nothing of boring .",0


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1. Embedding lookup
        embedded = self.embedding(input_ids)

        # 2. LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4. Apply dropout
        hidden = self.dropout(hidden)  

        # 5. Final classification layer (returns logits)
        output = self.fc(hidden)

        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    One epoch of training. 
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels' in each batch.
    - optimizer, criterion: training components (e.g., Adam, BCEWithLogitsLoss).
    - device: 'cpu' or 'cuda'.
    """
    model.train()
    losses = []
    correct_predictions = 0

    # For calculating precision, recall, F1:
    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 1) Forward pass -> raw logits
        logits = model(input_ids)  # shape: (batch_size, 1)
        logits = logits.squeeze(dim=1)  # shape: (batch_size,)

        # 2) Compute loss (BCEWithLogitsLoss expects raw logits)
        loss = criterion(logits, labels.float())

        # 3) Backprop + optimization
        loss.backward()
        optimizer.step()

        # 4) Track loss
        losses.append(loss.item())

        # 5) Convert logits -> probabilities -> predicted classes
        probs = torch.sigmoid(logits)          # in [0, 1]
        preds_cls = (probs >= 0.5).long()      # threshold at 0.5

        # 6) Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # 7) Collect for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate overall metrics for the epoch
    precision = precision_score(all_labels, all_preds, zero_division=0, average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


def eval_model(model, data_loader, criterion, device):
    """
    Evaluation function. Similar to train_epoch, but no backprop.
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels'.
    - criterion: e.g., BCEWithLogitsLoss for binary classification.
    - device: 'cpu' or 'cuda'.
    """
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            # 1) Forward pass -> logits
            logits = model(input_ids)  # shape: (batch_size, 1)
            logits = logits.squeeze(dim=1)  # shape: (batch_size,)

            # 2) Compute loss
            loss = criterion(logits, labels.float())
            losses.append(loss.item())

            # 3) Convert logits -> probabilities -> predicted classes
            probs = torch.sigmoid(logits)
            preds_cls = (probs >= 0.5).long()

            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    # Metrics
    precision = precision_score(all_labels, all_preds, zero_division=0, average='macro')
    recall = recall_score(all_labels, all_preds, zero_division=0, average='macro')
    f1 = f1_score(all_labels, all_preds, zero_division=0, average='macro')
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device)
        
        val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
            model, val_loader, criterion, device)
        
        print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
            f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
            f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')
    return train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1

In [10]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=['seed', 'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1',
                                'val_loss', 'val_acc', 'val_prec', 'val_rec', 'val_f1',
                                'test_loss', 'test_acc', 'test_prec', 'test_rec', 'test_f1',
                                'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
                                'max_memory_usage_test', 'max_vram_usage_test', 'total_time_test'])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1 = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1 = retval

    results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
                                                val_loss, val_acc, val_prec, val_rec, val_f1,
                                                test_loss, test_acc, test_prec, test_rec, test_f1,
                                                max_memory_usage_train, max_vram_usage_train, total_time_train,
                                                max_memory_usage_test, max_vram_usage_test, total_time_test]],
                                                columns=results.columns)], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6640, Accuracy: 0.5903, Precision: 0.5903, Recall: 0.5903, F1 Score: 0.5902
Val Loss: 0.6173, Accuracy: 0.6585, Precision: 0.6586, Recall: 0.6585, F1 Score: 0.6585
Epoch 2/5
Train Loss: 0.5534, Accuracy: 0.7299, Precision: 0.7306, Recall: 0.7299, F1 Score: 0.7297
Val Loss: 0.5780, Accuracy: 0.6829, Precision: 0.6930, Recall: 0.6829, F1 Score: 0.6787
Epoch 3/5
Train Loss: 0.4162, Accuracy: 0.8197, Precision: 0.8201, Recall: 0.8197, F1 Score: 0.8196
Val Loss: 0.6195, Accuracy: 0.7261, Precision: 0.7261, Recall: 0.7261, F1 Score: 0.7261
Epoch 4/5
Train Loss: 0.3068, Accuracy: 0.8781, Precision: 0.8785, Recall: 0.8781, F1 Score: 0.8780
Val Loss: 0.6530, Accuracy: 0.7158, Precision: 0.7172, Recall: 0.7158, F1 Score: 0.7153
Epoch 5/5
Train Loss: 0.2061, Accuracy: 0.9280, Precision: 0.9281, Recall: 0.9280, F1 Score: 0.9280
Val Loss: 0.8662, Accuracy: 0.7233, Precision: 0.7242, Recall: 0.7233, F1 Score: 0.7230


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_25400\3438238464.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6715, Accuracy: 0.5782, Precision: 0.5794, Recall: 0.5782, F1 Score: 0.5765
Val Loss: 0.6348, Accuracy: 0.6473, Precision: 0.6475, Recall: 0.6473, F1 Score: 0.6471
Epoch 2/5
Train Loss: 0.5582, Accuracy: 0.7208, Precision: 0.7220, Recall: 0.7208, F1 Score: 0.7204
Val Loss: 0.6004, Accuracy: 0.6754, Precision: 0.6848, Recall: 0.6754, F1 Score: 0.6713
Epoch 3/5
Train Loss: 0.4269, Accuracy: 0.8130, Precision: 0.8137, Recall: 0.8130, F1 Score: 0.8129
Val Loss: 0.5692, Accuracy: 0.7092, Precision: 0.7100, Recall: 0.7092, F1 Score: 0.7089
Epoch 4/5
Train Loss: 0.3118, Accuracy: 0.8758, Precision: 0.8763, Recall: 0.8758, F1 Score: 0.8758
Val Loss: 0.7210, Accuracy: 0.7083, Precision: 0.7096, Recall: 0.7083, F1 Score: 0.7078
Epoch 5/5
Train Loss: 0.2110, Accuracy: 0.9246, Precision: 0.9248, Recall: 0.9246, F1 Score: 0.9246
Val Loss: 0.7768, Accuracy: 0.7176, Precision: 0.7176, Recall: 0.7176, F1 Score: 0.7176


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6606, Accuracy: 0.6032, Precision: 0.6036, Recall: 0.6032, F1 Score: 0.6028
Val Loss: 0.6203, Accuracy: 0.6492, Precision: 0.6508, Recall: 0.6492, F1 Score: 0.6482
Epoch 2/5
Train Loss: 0.5369, Accuracy: 0.7417, Precision: 0.7419, Recall: 0.7417, F1 Score: 0.7417
Val Loss: 0.5824, Accuracy: 0.6895, Precision: 0.6896, Recall: 0.6895, F1 Score: 0.6894
Epoch 3/5
Train Loss: 0.4158, Accuracy: 0.8218, Precision: 0.8225, Recall: 0.8218, F1 Score: 0.8217
Val Loss: 0.5610, Accuracy: 0.7373, Precision: 0.7375, Recall: 0.7373, F1 Score: 0.7373
Epoch 4/5
Train Loss: 0.2974, Accuracy: 0.8829, Precision: 0.8830, Recall: 0.8829, F1 Score: 0.8829
Val Loss: 0.6093, Accuracy: 0.7402, Precision: 0.7438, Recall: 0.7402, F1 Score: 0.7392
Epoch 5/5
Train Loss: 0.1944, Accuracy: 0.9301, Precision: 0.9303, Recall: 0.9301, F1 Score: 0.9301
Val Loss: 0.7635, Accuracy: 0.7270, Precision: 0.7277, Recall: 0.7270, F1 Score: 0.7268


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:489: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_binary2.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,...,test_acc,test_prec,test_rec,test_f1,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.206097,0.928019,0.928148,0.928019,0.928013,0.866191,0.723265,0.724231,0.723265,...,0.747655,0.748727,0.747655,0.747382,1298.679688,231.628906,22.782322,1298.726562,194.781250,0.848279
1,3,0.210951,0.924619,0.924830,0.924619,0.924610,0.776778,0.717636,0.717637,0.717636,...,0.764540,0.764616,0.764540,0.764524,1299.312500,232.615234,22.657155,1299.308594,195.835938,0.769905
2,5,0.194353,0.930129,0.930273,0.930129,0.930123,0.763463,0.727017,0.727691,0.727017,...,0.747655,0.748040,0.747655,0.747557,1314.230469,231.113281,16.012360,1314.175781,194.156250,0.715096


In [12]:
torch.save(model.state_dict(), 'results/lstm_binary2.pth')